In [ ]:
from IPython.display import HTML, display

# We load an interactive HTML5 game canvas that mirrors the physics and rules of your proposal!
html_code = """
<div style="text-align: center; background-color: #222; padding: 15px; border-radius: 10px; font-family: sans-serif; color: white;">
    <h2>🏁 Ready, Set, Go! - Racing Tournament Arena</h2>
    <p>Use <b>Arrow Keys</b> or <b>WASD</b> to steer/drive | Hold <b>Spacebar</b> for Nitro Boost!</p>
    <canvas id="raceCanvas" width="820" height="560" style="border: 4px solid #444; border-radius: 5px; background-color: #2f6b3a;"></canvas>
    <div style="margin-top: 10px; font-size: 16px;">
        <span style="margin-right: 15px;"><b>Lap:</b> <span id="lapDisplay">1/3</span></span>
        <span style="margin-right: 15px;"><b>Speed:</b> <span id="speedDisplay">0</span> km/h</span>
        <span style="margin-right: 15px;"><b>Nitro:</b> <span id="nitroDisplay">100%</span></span>
        <span><b>Time:</b> <span id="timeDisplay">0.0</span>s</span>
    </div>
</div>

<script>
(function() {
    const canvas = document.getElementById('raceCanvas');
    const ctx = canvas.getContext('2d');

    // Track layout (Oval)
    const CX = 410, CY = 280, RX = 300, RY = 170, HALF = 45;

    // Player Car Setup
    let car = {
        x: CX + RX,
        y: CY,
        angle: -Math.PI / 2,
        speed: 0,
        maxSpeed: 5,
        laps: 1,
        nitro: 100,
        bonuses: 0,
        checkpointIndex: 0
    };

    // Keyboard Input State
    let keys = {};
    window.addEventListener('keydown', (e) => {
        if (["Space", "ArrowUp", "ArrowDown", "ArrowLeft", "ArrowRight"].indexOf(e.code) > -1) {
            e.preventDefault();
        }
        keys[e.code] = true;
    });
    window.addEventListener('keyup', (e) => {
        keys[e.code] = false;
    });

    // Generate 5 random golden bonuses on track
    let bonuses = [];
    for (let i = 0; i < 5; i++) {
        let angle = Math.random() * Math.PI * 2;
        bonuses.push({
            x: CX + RX * Math.cos(angle) + (Math.random() - 0.5) * 30,
            y: CY + RY * Math.sin(angle) + (Math.random() - 0.5) * 30
        });
    }

    let startTime = Date.now();

    function updatePhysics() {
        let steer = 0;
        if (keys['ArrowLeft'] || keys['KeyA']) steer = -1;
        if (keys['ArrowRight'] || keys['KeyD']) steer = 1;

        let accelerating = keys['ArrowUp'] || keys['KeyW'];
        let braking = keys['ArrowDown'] || keys['KeyS'];
        let nitroActive = keys['Space'] && car.nitro > 0;

        let topSpeed = car.maxSpeed;
        if (nitroActive && accelerating) {
            topSpeed += 2.5;
            car.nitro = Math.max(0, car.nitro - 0.5);
        }

        if (accelerating) {
            car.speed = Math.min(topSpeed, car.speed + 0.15);
        } else if (braking) {
            car.speed = Math.max(-2, car.speed - 0.3);
        } else {
            car.speed *= 0.98; // Friction
        }

        car.angle += steer * 0.04 * (Math.min(1, Math.abs(car.speed) / 2));
        car.x += Math.cos(car.angle) * car.speed;
        car.y += Math.sin(car.angle) * car.speed;

        // Boundary limits
        car.x = Math.max(15, Math.min(canvas.width - 15, car.x));
        car.y = Math.max(15, Math.min(canvas.height - 15, car.y));

        // Check Bonus collection
        bonuses.forEach((b) => {
            let d = Math.hypot(b.x - car.x, b.y - car.y);
            if (d < 20) {
                car.bonuses++;
                let angle = Math.random() * Math.PI * 2;
                b.x = CX + RX * Math.cos(angle) + (Math.random() - 0.5) * 30;
                b.y = CY + RY * Math.sin(angle) + (Math.random() - 0.5) * 30;
            }
        });

        // Lap system: Simple crossing logic near line
        if (Math.abs(car.x - (CX + RX)) < 40 && Math.abs(car.y - CY) < 40) {
            if (car.checkpointIndex === 1) {
                car.laps++;
                car.checkpointIndex = 0;
            }
        } else if (Math.abs(car.x - (CX - RX)) < 60) {
            car.checkpointIndex = 1; // Halfway
        }
    }

    function drawTrack() {
        ctx.fillStyle = '#2f6b3a';
        ctx.fillRect(0, 0, canvas.width, canvas.height);

        // Main track
        ctx.beginPath();
        ctx.ellipse(CX, CY, RX + HALF, RY + HALF, 0, 0, Math.PI * 2);
        ctx.fillStyle = '#3b3d44';
        ctx.fill();

        // Inner green field
        ctx.beginPath();
        ctx.ellipse(CX, CY, RX - HALF, RY - HALF, 0, 0, Math.PI * 2);
        ctx.fillStyle = '#2f6b3a';
        ctx.fill();

        // Center guide line
        ctx.beginPath();
        ctx.ellipse(CX, CY, RX, RY, 0, 0, Math.PI * 2);
        ctx.strokeStyle = '#555';
        ctx.lineWidth = 2;
        ctx.setLineDash([10, 15]);
        ctx.stroke();
        ctx.setLineDash([]);

        // Checkered Start Line
        ctx.fillStyle = 'white';
        ctx.fillRect(CX + RX - HALF, CY - 5, HALF * 2, 10);
    }

    function drawCar() {
        ctx.save();
        ctx.translate(car.x, car.y);
        ctx.rotate(car.angle);

        // Wheels
        ctx.fillStyle = '#111';
        ctx.fillRect(-12, -10, 6, 4);
        ctx.fillRect(6, -10, 6, 4);
        ctx.fillRect(-12, 6, 6, 4);
        ctx.fillRect(6, 6, 6, 4);

        // Car Body
        ctx.fillStyle = '#d7261e';
        ctx.fillRect(-15, -7, 30, 14);

        // Cabin
        ctx.fillStyle = '#7bc8f6';
        ctx.fillRect(-3, -4, 10, 8);

        ctx.restore();
    }

    function drawBonuses() {
        bonuses.forEach((b) => {
            ctx.beginPath();
            ctx.arc(b.x, b.y, 8, 0, Math.PI * 2);
            ctx.fillStyle = 'gold';
            ctx.fill();
            ctx.lineWidth = 2;
            ctx.strokeStyle = '#b38600';
            ctx.stroke();
        });
    }

    function gameLoop() {
        updatePhysics();
        drawTrack();
        drawBonuses();
        drawCar();

        // Dashboard Updates
        document.getElementById('lapDisplay').innerText = car.laps + "/3";
        document.getElementById('speedDisplay').innerText = Math.round(Math.abs(car.speed) * 30);
        document.getElementById('nitroDisplay').innerText = Math.round(car.nitro) + "%";
        document.getElementById('timeDisplay').innerText = ((Date.now() - startTime) / 1000).toFixed(1);

        requestAnimationFrame(gameLoop);
    }

    gameLoop();
})();
</script>
"""
display(HTML(html_code))
